# SVM (Support Vector Machine)

## ¿Cuál es el objetivo de SVM?

Las **Máquinas de Vectores de Soporte (SVM)** son un algoritmo de **clasificación** cuyo objetivo es encontrar el **hiperplano** que mejor separa dos clases, maximizando el **margen** entre ellas.

¿Qué significa esto? Dados datos de dos clases, existen infinitos hiperplanos que podrían separarlos. SVM elige el que tiene la **mayor distancia** a los puntos más cercanos de cada clase. Estos puntos más cercanos se llaman **vectores de soporte** (de ahí el nombre).

El hiperplano de decisión se define como:

$$
\mathbf{w} \cdot \mathbf{x} - b = 0
$$

Y la regla de clasificación es:

$$
\hat{y} = \text{sign}(\mathbf{w} \cdot \mathbf{x} - b) = \begin{cases} +1 & \text{si } \mathbf{w} \cdot \mathbf{x} - b > 0 \\ -1 & \text{si } \mathbf{w} \cdot \mathbf{x} - b < 0 \end{cases}
$$

Donde:

- $\mathbf{w}$: vector normal al hiperplano (define su orientación).
- $b$: sesgo (bias), desplaza el hiperplano respecto al origen.
- **Margen**: distancia entre los dos hiperplanos paralelos que pasan por los vectores de soporte.

**¿Por qué maximizar el margen?** Un margen más amplio significa mayor robustez: el clasificador generaliza mejor a datos nuevos porque hay más "espacio de seguridad" entre las clases.

**Nota:** En SVM las clases se codifican como $\{-1, +1\}$ (no $\{0, 1\}$ como en Regresión Logística).

**Supongamos un conjunto de datos simples**

Datos en 2D con dos clases linealmente separables:

| $x_1$ | $x_2$ | Clase ($y$) |
|-------|-------|------|
| 1     | 3     | -1   |
| 2     | 3     | -1   |
| 1     | 2     | -1   |
| 3     | 1     | -1   |
| 5     | 5     | +1   |
| 6     | 5     | +1   |
| 6     | 6     | +1   |
| 7     | 5     | +1   |

Hay muchas líneas que podrían separar estas dos clases. SVM encuentra la línea con el **margen más amplio**.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Datos de ejemplo
X = np.array([[1, 3], [2, 3], [1, 2], [3, 1],
              [5, 5], [6, 5], [6, 6], [7, 5]])
y = np.array([-1, -1, -1, -1, 1, 1, 1, 1])

plt.figure(figsize=(8, 6))
plt.scatter(X[y == -1, 0], X[y == -1, 1], color='blue', s=100, label='Clase -1', edgecolors='black')
plt.scatter(X[y == 1, 0], X[y == 1, 1], color='red', s=100, label='Clase +1', edgecolors='black')

# Dibujar varias líneas posibles (para ilustrar que hay muchas)
x_line = np.linspace(0, 8, 100)
plt.plot(x_line, -x_line + 8, 'g--', alpha=0.3, label='Posible separador 1')
plt.plot(x_line, -0.8*x_line + 7, 'purple', alpha=0.3, linestyle='--', label='Posible separador 2')
plt.plot(x_line, -1.2*x_line + 8.5, 'orange', alpha=0.3, linestyle='--', label='Posible separador 3')

plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.title('¿Cuál línea es la mejor? SVM elige la de mayor margen')
plt.xlim(0, 8)
plt.ylim(0, 8)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## ¿Cómo encuentra SVM el hiperplano óptimo?

SVM plantea el problema como una **optimización**: encontrar $\mathbf{w}$ y $b$ tales que el **margen sea máximo** y todos los puntos estén **correctamente clasificados**.

Los puntos que están exactamente sobre el borde del margen son los **vectores de soporte**. Son los puntos más "difíciles" de clasificar y los únicos que importan para definir el hiperplano. Si removeremos cualquier otro punto, el hiperplano no cambiaría.

## Demostración Matemática

### Paso 1: El margen y las restricciones

El hiperplano de decisión es $\mathbf{w} \cdot \mathbf{x} - b = 0$. Definimos dos hiperplanos paralelos que pasan por los vectores de soporte:

$$
\mathbf{w} \cdot \mathbf{x} - b = +1 \quad \text{(hiperplano positivo)}
$$
$$
\mathbf{w} \cdot \mathbf{x} - b = -1 \quad \text{(hiperplano negativo)}
$$

La **distancia** entre estos dos hiperplanos (el margen) es:

$$
\text{margen} = \frac{2}{\|\mathbf{w}\|}
$$

**Derivación de la fórmula del margen:**

La distancia de un punto $\mathbf{x}_0$ a un hiperplano $\mathbf{w} \cdot \mathbf{x} - b = 0$ es $\frac{|\mathbf{w} \cdot \mathbf{x}_0 - b|}{\|\mathbf{w}\|}$.

Para un vector de soporte positivo ($\mathbf{w} \cdot \mathbf{x} - b = +1$), su distancia al hiperplano de decisión es $\frac{1}{\|\mathbf{w}\|}$. Para un vector de soporte negativo ($\mathbf{w} \cdot \mathbf{x} - b = -1$), también es $\frac{1}{\|\mathbf{w}\|}$. La distancia total (margen) es $\frac{2}{\|\mathbf{w}\|}$.

Las **restricciones** para clasificación correcta son:

$$
y_i(\mathbf{w} \cdot \mathbf{x}_i - b) \geq 1 \quad \forall i
$$

Esto garantiza que:
- Si $y_i = +1$: $\mathbf{w} \cdot \mathbf{x}_i - b \geq +1$ (el punto está del lado positivo).
- Si $y_i = -1$: $\mathbf{w} \cdot \mathbf{x}_i - b \leq -1$ (el punto está del lado negativo).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(9, 7))

X = np.array([[1, 3], [2, 3], [1, 2], [3, 1],
              [5, 5], [6, 5], [6, 6], [7, 5]])
y = np.array([-1, -1, -1, -1, 1, 1, 1, 1])

ax.scatter(X[y == -1, 0], X[y == -1, 1], color='blue', s=100, edgecolors='black', zorder=5, label='Clase -1')
ax.scatter(X[y == 1, 0], X[y == 1, 1], color='red', s=100, edgecolors='black', zorder=5, label='Clase +1')

# Hiperplano de decisión y márgenes (aproximados para visualización)
x_line = np.linspace(0, 8, 100)
# Línea de decisión: w·x - b = 0
ax.plot(x_line, -x_line + 7.5, 'k-', linewidth=2, label='Hiperplano de decisión (w·x - b = 0)')
# Márgenes: w·x - b = ±1
ax.plot(x_line, -x_line + 8.8, 'k--', linewidth=1, alpha=0.7, label='w·x - b = +1')
ax.plot(x_line, -x_line + 6.2, 'k--', linewidth=1, alpha=0.7, label='w·x - b = -1')

# Sombrear margen
ax.fill_between(x_line, -x_line + 6.2, -x_line + 8.8, alpha=0.1, color='yellow')

# Marcar vectores de soporte (puntos más cercanos al hiperplano)
ax.scatter([3, 5], [1, 5], s=250, facecolors='none', edgecolors='green', linewidths=3,
           label='Vectores de soporte', zorder=6)

# Flecha del margen
ax.annotate('', xy=(4.5, 4.3), xytext=(3.2, 3.0),
            arrowprops=dict(arrowstyle='<->', color='green', lw=2))
ax.annotate('Margen = $\\frac{2}{||w||}$', xy=(2.5, 3.8), fontsize=12, color='green',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('SVM: Hiperplano de decisión, márgenes y vectores de soporte')
ax.set_xlim(0, 8)
ax.set_ylim(0, 8)
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left', fontsize=9)
plt.show()

### Paso 2: El problema de optimización

Queremos **maximizar el margen** $\frac{2}{\|\mathbf{w}\|}$ sujeto a que todos los puntos estén bien clasificados. Maximizar $\frac{2}{\|\mathbf{w}\|}$ es equivalente a **minimizar** $\frac{1}{2}\|\mathbf{w}\|^2$.

El problema de optimización original (**hard-margin SVM**) es:

$$
\min_{\mathbf{w}, b} \frac{1}{2} \|\mathbf{w}\|^2 \quad \text{sujeto a} \quad y_i(\mathbf{w} \cdot \mathbf{x}_i - b) \geq 1 \quad \forall i
$$

En la práctica, los datos rara vez son perfectamente separables. La versión **soft-margin** permite errores de clasificación, penalizándolos con un parámetro $\lambda$ (regularización).

La función de costo se convierte en la **Hinge Loss** más un término de regularización:

$$
J(\mathbf{w}, b) = \lambda \|\mathbf{w}\|^2 + \frac{1}{n}\sum_{i=1}^{n} \max(0, \ 1 - y_i(\mathbf{w} \cdot \mathbf{x}_i - b))
$$

Donde:

- $\lambda \|\mathbf{w}\|^2$: **regularización** (controla el tamaño del margen).
- $\max(0, 1 - y_i(\mathbf{w} \cdot \mathbf{x}_i - b))$: **hinge loss** (penaliza puntos mal clasificados o dentro del margen).

### Paso 3: La Hinge Loss

La **hinge loss** para un solo punto es:

$$
\ell_i = \max(0, \ 1 - y_i(\mathbf{w} \cdot \mathbf{x}_i - b))
$$

Su comportamiento:

- Si $y_i(\mathbf{w} \cdot \mathbf{x}_i - b) \geq 1$: el punto está **correctamente clasificado y fuera del margen** → $\ell_i = 0$ (sin penalización).
- Si $0 < y_i(\mathbf{w} \cdot \mathbf{x}_i - b) < 1$: el punto está **correctamente clasificado pero dentro del margen** → $\ell_i > 0$ (penalización proporcional).
- Si $y_i(\mathbf{w} \cdot \mathbf{x}_i - b) < 0$: el punto está **mal clasificado** → $\ell_i > 1$ (penalización alta).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

z = np.linspace(-2, 3, 300)
hinge = np.maximum(0, 1 - z)

plt.figure(figsize=(8, 5))
plt.plot(z, hinge, 'b-', linewidth=2, label='Hinge Loss: max(0, 1 - z)')
plt.axvline(x=1, color='red', linestyle='--', alpha=0.5, label='z = 1 (borde del margen)')
plt.axvline(x=0, color='gray', linestyle=':', alpha=0.5, label='z = 0 (hiperplano de decisión)')

# Regiones
plt.fill_between(z, 0, hinge, where=(z < 0), alpha=0.1, color='red')
plt.fill_between(z, 0, hinge, where=((z >= 0) & (z < 1)), alpha=0.1, color='orange')

plt.annotate('Mal clasificado\n(loss alto)', xy=(-1.5, 2.3), fontsize=10, ha='center',
             bbox=dict(boxstyle='round', facecolor='lightsalmon', alpha=0.7))
plt.annotate('Dentro del margen\n(loss > 0)', xy=(0.5, 0.8), fontsize=10, ha='center',
             bbox=dict(boxstyle='round', facecolor='moccasin', alpha=0.7))
plt.annotate('Fuera del margen\n(loss = 0)', xy=(2.2, 0.3), fontsize=10, ha='center',
             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))

plt.xlabel('$z = y_i(\mathbf{w} \cdot \mathbf{x}_i - b)$')
plt.ylabel('Loss')
plt.title('Hinge Loss')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### Paso 4: Gradientes para descenso de gradiente

Para minimizar la función de costo con **descenso de gradiente**, necesitamos los gradientes. Los calculamos por casos, según si el punto satisface o no la condición del margen.

**Caso 1: $y_i(\mathbf{w} \cdot \mathbf{x}_i - b) \geq 1$** (correctamente clasificado, fuera del margen)

La hinge loss es 0, así que solo queda el término de regularización:

$$
\frac{\partial J}{\partial \mathbf{w}} = 2\lambda \mathbf{w}
$$
$$
\frac{\partial J}{\partial b} = 0
$$

**Caso 2: $y_i(\mathbf{w} \cdot \mathbf{x}_i - b) < 1$** (dentro del margen o mal clasificado)

Tanto la regularización como la hinge loss contribuyen:

$$
\frac{\partial J}{\partial \mathbf{w}} = 2\lambda \mathbf{w} - y_i \mathbf{x}_i
$$
$$
\frac{\partial J}{\partial b} = y_i
$$

La **regla de actualización** en cada iteración es:

$$
\mathbf{w} \leftarrow \mathbf{w} - \alpha \cdot \frac{\partial J}{\partial \mathbf{w}}
$$
$$
b \leftarrow b - \alpha \cdot \frac{\partial J}{\partial b}
$$

Donde $\alpha$ es el learning rate.

### El parámetro de regularización $\lambda$

El parámetro $\lambda$ controla el balance entre:

- **Margen amplio** ($\lambda$ grande): margen más ancho, pero posiblemente más errores de clasificación.
- **Pocos errores** ($\lambda$ pequeño): clasifica mejor los datos de entrenamiento, pero con margen más estrecho (riesgo de sobreajuste).

En la práctica, se prueba con validación cruzada para encontrar el valor óptimo.

### Resumen del algoritmo

**Entrenamiento:**

1. Convertir etiquetas a $\{-1, +1\}$.
2. Inicializar $\mathbf{w} = \mathbf{0}$, $b = 0$.
3. Para cada iteración y cada punto $(\mathbf{x}_i, y_i)$:
   - Si $y_i(\mathbf{w} \cdot \mathbf{x}_i - b) \geq 1$:
     - $\mathbf{w} \leftarrow \mathbf{w} - \alpha \cdot 2\lambda\mathbf{w}$
   - Si no:
     - $\mathbf{w} \leftarrow \mathbf{w} - \alpha(2\lambda\mathbf{w} - y_i\mathbf{x}_i)$
     - $b \leftarrow b - \alpha \cdot y_i$

**Predicción:**

$$
\hat{y} = \text{sign}(\mathbf{w} \cdot \mathbf{x} - b)
$$

### Ejemplo numérico

Usemos un ejemplo simplificado con **2 features** y **4 muestras**.

| $x_1$ | $x_2$ | $y$ |
|-------|-------|-----|
| 1     | 2     | -1  |
| 2     | 1     | -1  |
| 4     | 5     | +1  |
| 5     | 4     | +1  |

Inicializamos: $\mathbf{w} = (0, 0)$, $b = 0$, $\alpha = 0.01$, $\lambda = 0.01$

**Iteración 1, punto 1:** $\mathbf{x}_1 = (1, 2)$, $y_1 = -1$

$$
y_1(\mathbf{w} \cdot \mathbf{x}_1 - b) = -1 \cdot (0 \cdot 1 + 0 \cdot 2 - 0) = 0
$$

Como $0 < 1$, estamos en el **Caso 2** (dentro del margen):

$$
\mathbf{w} \leftarrow (0, 0) - 0.01 \cdot (2 \cdot 0.01 \cdot (0, 0) - (-1) \cdot (1, 2)) = (0, 0) - 0.01 \cdot (1, 2) = (-0.01, -0.02)
$$

$$
b \leftarrow 0 - 0.01 \cdot (-1) = 0.01
$$

Después de una sola muestra, $\mathbf{w}$ ya apunta hacia el lado negativo (componentes negativas porque $y = -1$), lo cual es el comportamiento correcto. El proceso se repite miles de veces hasta que los pesos convergen.

In [ ]:
import numpy as np

# Datos de ejemplo
X_ej = np.array([[1, 2], [2, 1], [4, 5], [5, 4]], dtype=np.float64)
y_ej = np.array([-1, -1, 1, 1])

w = np.zeros(2)
b = 0.0
lr = 0.01
lam = 0.01

print("=== Ejemplo numérico: SVM paso a paso ===")
print(f"Datos: X = {X_ej.tolist()}, y = {y_ej.tolist()}")
print(f"Parámetros iniciales: w = {w}, b = {b}, lr = {lr}, λ = {lam}\n")

for it in range(3):
    print(f"--- Iteración {it + 1} ---")
    for idx, (x_i, y_i) in enumerate(zip(X_ej, y_ej)):
        condition_val = y_i * (np.dot(x_i, w) - b)
        condition = condition_val >= 1

        if condition:
            dw = 2 * lam * w
            db = 0
            caso = "Caso 1 (fuera del margen)"
        else:
            dw = 2 * lam * w - y_i * x_i
            db = y_i
            caso = "Caso 2 (dentro del margen)"

        w = w - lr * dw
        b = b - lr * db

        print(f"  x={x_i}, y={y_i:+d}: y(w·x-b)={condition_val:+.4f} → {caso}")
        print(f"    w = ({w[0]:.4f}, {w[1]:.4f}), b = {b:.4f}")
    print()

## Implementación con Código

### Funciones

In [ ]:
import numpy as np

def svm_fit(X, y, learning_rate=0.001, lambda_param=0.01, n_iters=1000):
    """Entrena un SVM lineal con descenso de gradiente"""
    n_samples, n_features = X.shape
    y_ = np.where(y <= 0, -1, 1)

    w = np.zeros(n_features)
    b = 0

    for _ in range(n_iters):
        for idx, x_i in enumerate(X):
            condition = y_[idx] * (np.dot(x_i, w) - b) >= 1
            if condition:
                w -= learning_rate * (2 * lambda_param * w)
            else:
                w -= learning_rate * (2 * lambda_param * w - np.dot(x_i, y_[idx]))
                b -= learning_rate * y_[idx]

    return w, b

def svm_predict(X, w, b):
    """Predice clases (-1 o +1)"""
    approx = np.dot(X, w) - b
    return np.sign(approx)

In [ ]:
import matplotlib.pyplot as plt
from sklearn import datasets

# Generar datos
X, y = datasets.make_blobs(
    n_samples=50, n_features=2, centers=2, cluster_std=1.05, random_state=40
)
y = np.where(y == 0, -1, 1)

# Entrenar
w, b = svm_fit(X, y)
print(f"w = {w}")
print(f"b = {b}")

# Predecir y evaluar
predictions = svm_predict(X, w, b)
accuracy = np.sum(y == predictions) / len(y)
print(f"Accuracy: {accuracy:.4f}")

# Visualizar
def get_hyperplane_value(x, w, b, offset):
    return (-w[0] * x + b + offset) / w[1]

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(X[:, 0], X[:, 1], marker='o', c=y, cmap='bwr', s=60, edgecolors='black')

x0_1 = np.amin(X[:, 0])
x0_2 = np.amax(X[:, 0])

# Hiperplano de decisión
ax.plot([x0_1, x0_2], [get_hyperplane_value(x0_1, w, b, 0), get_hyperplane_value(x0_2, w, b, 0)],
        'y--', linewidth=2, label='Hiperplano de decisión')
# Márgenes
ax.plot([x0_1, x0_2], [get_hyperplane_value(x0_1, w, b, -1), get_hyperplane_value(x0_2, w, b, -1)],
        'k-', linewidth=1, label='Margen -1')
ax.plot([x0_1, x0_2], [get_hyperplane_value(x0_1, w, b, 1), get_hyperplane_value(x0_2, w, b, 1)],
        'k-', linewidth=1, label='Margen +1')

ax.set_ylim([np.amin(X[:, 1]) - 3, np.amax(X[:, 1]) + 3])
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('SVM: Hiperplano y márgenes')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

### Clase

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


class SVM:
    def __init__(self, learning_rate=0.001, lambda_param=0.01, n_iters=1000):
        self.lr = learning_rate
        self.lambda_param = lambda_param
        self.n_iters = n_iters
        self.w = None
        self.b = None
        self.losses = []

    def fit(self, X, y):
        """Entrena el SVM con descenso de gradiente"""
        n_samples, n_features = X.shape
        y_ = np.where(y <= 0, -1, 1)

        self.w = np.zeros(n_features)
        self.b = 0
        self.losses = []

        for _ in range(self.n_iters):
            for idx, x_i in enumerate(X):
                condition = y_[idx] * (np.dot(x_i, self.w) - self.b) >= 1
                if condition:
                    self.w -= self.lr * (2 * self.lambda_param * self.w)
                else:
                    self.w -= self.lr * (
                        2 * self.lambda_param * self.w - np.dot(x_i, y_[idx])
                    )
                    self.b -= self.lr * y_[idx]

            # Registrar loss
            loss = self._compute_loss(X, y_)
            self.losses.append(loss)

    def predict(self, X):
        """Predice clases (-1 o +1)"""
        approx = np.dot(X, self.w) - self.b
        return np.sign(approx)

    def score(self, X, y):
        """Calcula la accuracy del modelo"""
        y_ = np.where(y <= 0, -1, 1)
        predictions = self.predict(X)
        return np.sum(predictions == y_) / len(y_)

    def _compute_loss(self, X, y):
        """Calcula la hinge loss + regularización"""
        distances = 1 - y * (np.dot(X, self.w) - self.b)
        hinge_loss = np.mean(np.maximum(0, distances))
        return self.lambda_param * np.dot(self.w, self.w) + hinge_loss

    def plot_loss(self):
        """Grafica la evolución del loss durante el entrenamiento"""
        plt.figure(figsize=(8, 5))
        plt.plot(self.losses, 'b-', linewidth=1)
        plt.xlabel('Iteración')
        plt.ylabel('Loss (Hinge + Regularización)')
        plt.title('Evolución del Loss durante el entrenamiento')
        plt.grid(True, alpha=0.3)
        plt.show()

    def plot_decision_boundary(self, X, y):
        """Visualiza el hiperplano, márgenes y vectores de soporte"""
        y_ = np.where(y <= 0, -1, 1)

        def get_hyperplane_value(x, w, b, offset):
            return (-w[0] * x + b + offset) / w[1]

        fig, ax = plt.subplots(figsize=(10, 7))
        ax.scatter(X[:, 0], X[:, 1], marker='o', c=y_, cmap='bwr', s=60, edgecolors='black')

        x0_1 = np.amin(X[:, 0])
        x0_2 = np.amax(X[:, 0])

        # Hiperplano de decisión
        ax.plot([x0_1, x0_2],
                [get_hyperplane_value(x0_1, self.w, self.b, 0),
                 get_hyperplane_value(x0_2, self.w, self.b, 0)],
                'y--', linewidth=2, label='Hiperplano de decisión')

        # Márgenes
        ax.plot([x0_1, x0_2],
                [get_hyperplane_value(x0_1, self.w, self.b, -1),
                 get_hyperplane_value(x0_2, self.w, self.b, -1)],
                'k-', linewidth=1, label='Margen -1')
        ax.plot([x0_1, x0_2],
                [get_hyperplane_value(x0_1, self.w, self.b, 1),
                 get_hyperplane_value(x0_2, self.w, self.b, 1)],
                'k-', linewidth=1, label='Margen +1')

        # Sombrear margen
        x_fill = np.linspace(x0_1, x0_2, 100)
        y_m = [get_hyperplane_value(x, self.w, self.b, -1) for x in x_fill]
        y_p = [get_hyperplane_value(x, self.w, self.b, 1) for x in x_fill]
        ax.fill_between(x_fill, y_m, y_p, alpha=0.1, color='yellow')

        # Marcar vectores de soporte (puntos cerca del margen)
        distances = y_ * (np.dot(X, self.w) - self.b)
        sv_mask = (distances >= 0.9) & (distances <= 1.1)
        if sv_mask.any():
            ax.scatter(X[sv_mask, 0], X[sv_mask, 1], s=200, facecolors='none',
                       edgecolors='green', linewidths=2, label='Vectores de soporte')

        ax.set_ylim([np.amin(X[:, 1]) - 3, np.amax(X[:, 1]) + 3])
        ax.set_xlabel('$x_1$')
        ax.set_ylabel('$x_2$')
        ax.set_title('SVM: Hiperplano, márgenes y vectores de soporte')
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.show()

In [ ]:
from sklearn import datasets

# Generar datos
X, y = datasets.make_blobs(
    n_samples=50, n_features=2, centers=2, cluster_std=1.05, random_state=40
)
y = np.where(y == 0, -1, 1)

# Crear y entrenar modelo
model = SVM(learning_rate=0.001, lambda_param=0.01, n_iters=1000)
model.fit(X, y)

# Evaluar
acc = model.score(X, y)
print(f"w = {model.w}")
print(f"b = {model.b:.4f}")
print(f"Accuracy: {acc:.4f}")

# Visualizar loss y frontera
model.plot_loss()
model.plot_decision_boundary(X, y)

In [ ]:
# Comparar diferentes valores de λ (regularización)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

lambdas = [0.001, 0.01, 0.1]
for ax, lam in zip(axes, lambdas):
    model_comp = SVM(learning_rate=0.001, lambda_param=lam, n_iters=1000)
    model_comp.fit(X, y)
    acc = model_comp.score(X, y)

    def get_hp(x, w, b, off):
        return (-w[0] * x + b + off) / w[1]

    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='bwr', s=40, edgecolors='black')

    x0_1, x0_2 = np.amin(X[:, 0]), np.amax(X[:, 0])
    ax.plot([x0_1, x0_2], [get_hp(x0_1, model_comp.w, model_comp.b, 0),
                            get_hp(x0_2, model_comp.w, model_comp.b, 0)], 'y--', linewidth=2)
    ax.plot([x0_1, x0_2], [get_hp(x0_1, model_comp.w, model_comp.b, -1),
                            get_hp(x0_2, model_comp.w, model_comp.b, -1)], 'k-', linewidth=1)
    ax.plot([x0_1, x0_2], [get_hp(x0_1, model_comp.w, model_comp.b, 1),
                            get_hp(x0_2, model_comp.w, model_comp.b, 1)], 'k-', linewidth=1)

    x_fill = np.linspace(x0_1, x0_2, 100)
    ax.fill_between(x_fill,
                    [get_hp(x, model_comp.w, model_comp.b, -1) for x in x_fill],
                    [get_hp(x, model_comp.w, model_comp.b, 1) for x in x_fill],
                    alpha=0.1, color='yellow')

    ax.set_ylim([np.amin(X[:, 1]) - 3, np.amax(X[:, 1]) + 3])
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.set_title(f'$\lambda$ = {lam} (acc = {acc:.2f})')
    ax.grid(True, alpha=0.3)

plt.suptitle('Efecto de la regularización $\lambda$ en SVM', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()